# H3 · `scripts/qa_review.py`

## What this file is for

The harness checking its own results, rather than assuming a run that completed was a run that was
right. Four passes were proposed; this file holds the two that are built.

**Pass 3** re-derives every `NOT APPLICABLE` in the stored runs from ISO's own CSVs, independently
of the code that produced the refusal in the first place. **Pass 4** assembles adversarial briefs --
one prompt per specialist agent, each told to refute a claim rather than confirm it -- because a
Python script cannot invoke those agents itself; dispatching them is a person's job.

**Why pass 3 matters more than it sounds like it should:** `NOT APPLICABLE` is the one outcome never
counted as a failure, so it is the one place a defect can sit indefinitely. On 2026-08-14 twenty
jurisdictions reported `NOT APPLICABLE` for terrorism with a readable, wrong reason -- our own inert
`ZipCode` fallback, not ISO's filing (OI-91). A refusal with a well-written explanation is still a
refusal, and nothing had been checking the explanation.

**Depends on:** [`H1 variants.py`](01-variants.ipynb) for what a control means,
[`H5 runstore.py`](05-runstore.ipynb) for the runs it reviews.

## Its public surface

Generated from the module, so it can't drift.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0, str(Path.cwd().parent.parent / "scripts"))

import inspect
import qa_review as qr

for name, obj in vars(qr).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != qr.__name__:
        continue
    if inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        v = repr(obj)
        print(f"{name} = {v if len(v) < 90 else v[:87] + '...'}")

## The smallest thing that works

`review_not_applicable` takes a jurisdiction, a configuration and re-checks it against ISO's own
field and domain CSVs -- reading the files directly, never calling `variants.build` or
`Declared.values`. Re-deriving the same answer with the same code would prove nothing.

In [ ]:
v = qr.review_not_applicable("AK", {"locations": 2}, "AK declares 1 territory")
print("verdict:", v["verdict"])
for f in v["findings"]:
    print(" ", f)

`CONFIRMED` here means ISO's own rate-table CSVs, read fresh, agree AK files exactly one prem/ops
territory -- the refusal holds up under an independent check.

## The interesting case

### One `CONFIRMED` settles the whole configuration -- and that rule was earned, not assumed

A `NOT APPLICABLE` means *at least one* control in the configuration cannot be expressed. Every
other control being perfectly legal is the ordinary case, not evidence of anything -- so aggregating
worst-first (one `CONFIRMED` wins, regardless of what else is in the mix) is correct. Aggregating any
other way produced **20+ false findings on the first run**: Montana was reported as wrongly refusing
a limit it declares fine, when the real cause was `locations=2` against its single territory.

In [ ]:
# occurrence_limit is legal in AK (CONTRADICTED if it were the whole story);
# locations=2 is not. The worst-first rule reports CONFIRMED for the pair.
v = qr.review_not_applicable(
    "AK", {"occurrence_limit": "1,000,000 CSL", "locations": 2}, "r")
print("overall:", v["verdict"])
for f in v["findings"]:
    print(" ", f["control"], "->", f["verdict"])

### Pass 4 does not run agents -- it writes their instructions

`brief()` is a pure function: given a claim and the evidence, it returns one prompt per reviewer,
each told to see only its own source and to **refute**, never confirm. Nothing here dispatches
anything; a person reads the prompts and sends them.

In [ ]:
b = qr.brief(
    "Our premium for OK is correct and ISO differs for a reason we understand.",
    {"jurisdiction": "OK", "our premium": 8816, "ISO premium": "refused"},
    "Which of the two is right, and which filed rule decides it?")
print("reviewers:", list(b["prompts"]))
print()
print(b["prompts"]["gl-authority"][:280], "...")

## What it refuses

Not every refusal traces back to a declared domain table -- some come from a condition inside an
applier (`classifications` checking whether enough distinct class codes exist, say). Pass 3 will not
call that `CONFIRMED` on the strength of a guess: it returns `UNVERIFIED` and says exactly what
would settle it, rather than a false `CONFIRMED` that reads as certainty it does not have.

In [ ]:
v = qr.review_not_applicable("AK", {"classifications": 30}, "r")
print(v["verdict"])
print(" ", v["findings"][0]["why"])

## Try it yourself

1. `qr.classify(reason)` sorts a failure into `ISO_QUESTION` or `LOCAL_PROBLEM` off a short list of
   substrings in `_LOCAL_MARKERS`. What happens to a reason that matches none of them -- and why is
   that the safe default?
2. Build a configuration you know is legal everywhere (say, just `exposure`) and confirm
   `review_not_applicable` never gets called for it -- `NOT APPLICABLE` only exists to review, not
   every result.
3. `export_refusals` writes the exact payload behind a refusal to `results/refused-payloads/`, ready
   to send to ISO. Read its docstring for why a refused-before-calling payload is *self-concealing*
   evidence -- the fix that stops us wasting a call also stops us learning what ISO would have said.

In [ ]:
# your turn